# Notebook 00 (Participant): Setup + API Warmup

This chapter introduces the benchmark interface itself.
Your target is to read a problem definition as a reproducible scientific object, not just an API object.


**Edit-safe start:** this notebook opens from GitHub in read-only source mode. Use **File -> Save a copy in Drive** before running edits so your changes stay in your own workspace.


## Notebook map

This notebook is written as a standalone lab chapter:
- context first,
- implementation second,
- interpretation third.

If you are following asynchronously, run cells in order and use the success checks to validate each stage before moving on.

### Public exercise legend
- `PUBLIC FILL-IN CELL`: this is the part you edit during the workshop.
- `CHECKPOINT`: run immediately after your edits; if it fails, fix before moving on.
- `IF YOU ARE STUCK`: use the hint comments in that same cell (do not jump ahead).


## Standalone guide

Learning goals:
- identify what EngiBench fixes (problem contract),
- identify what researchers can vary (methods),
- inspect conditions, objectives, and constraints with reproducibility in mind.


## Why this warmup matters

In engineering-design ML, many apparent gains come from hidden evaluation differences.
This warmup is about controlling that risk: understanding exactly what is held constant by the benchmark.


## Optional install cell (fresh Colab)

Run this cell only on a fresh Colab runtime or if imports fail.
Local environments can usually skip it.


In [ ]:
# Colab/local dependency bootstrap
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
FORCE_INSTALL = False  # Set True to force install outside Colab


def pip_install(packages: list[str]):
    cmd = [sys.executable, "-m", "pip", "install", *packages]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)


BASE_PACKAGES = ["engibench[beams2d]", "matplotlib", "seaborn"]

if IN_COLAB or FORCE_INSTALL:
    print("Installing dependencies...")
    pip_install(BASE_PACKAGES)

    try:
        import torch  # noqa: F401
    except Exception:
        pip_install(["torch", "torchvision"])

    print("Dependency install complete.")
else:
    print("Skipping install (using current environment). Set FORCE_INSTALL=True to install here.")

### Step 1 - Initialize reproducible session

Set seed and print versions.
If this step is inconsistent across machines, downstream comparisons are not interpretable.


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

import engibench
from engibench.problems.beams2d.v0 import Beams2D

SEED = 7
random.seed(SEED)
np.random.seed(SEED)

print("engibench version:", engibench.__version__)
print("seed:", SEED)

### Step 2 - Instantiate the benchmark problem (PUBLIC FILL-IN)

Create `Beams2D` and inspect its key fields.

What this step teaches:
- how a benchmark problem defines design space + objectives + conditions,
- which fields are part of the public contract you must preserve for fair comparison.

Success criteria:
- you can name each contract field and explain its role.


In [ ]:
# PUBLIC FILL-IN CELL 00-A
# Goal: instantiate the benchmark problem and inspect the full contract.

# START FILL ---------------------------------------------------------------
problem = None  # Example: Beams2D(seed=SEED)
# END FILL -----------------------------------------------------------------

if problem is None:
    raise RuntimeError("Set `problem` before running this cell (example: Beams2D(seed=SEED)).")

print("Problem class:", type(problem).__name__)
print("Design space:", problem.design_space)
print("Objectives:", problem.objectives)
print("Conditions instance:", problem.conditions)
print("Condition keys:", problem.conditions_keys)
print("Dataset ID:", problem.dataset_id)

# CHECKPOINT
assert hasattr(problem, "design_space"), "Problem is missing design_space"
assert hasattr(problem, "objectives"), "Problem is missing objectives"
assert len(problem.conditions_keys) > 0, "conditions_keys should not be empty"
print("Checkpoint passed: problem contract is visible and ready.")

### Step 3 - Inspect dataset structure (PUBLIC FILL-IN)

Load one train sample and inspect keys + shapes.

What this step teaches:
- how conditions and designs are stored in the benchmark dataset,
- what must be serialized later when handing off artifacts between notebooks.

Success criteria:
- `design` has the expected spatial shape,
- `config` contains exactly the condition keys used by the problem.


In [ ]:
# PUBLIC FILL-IN CELL 00-B
# Goal: inspect one training sample and build a valid config dictionary.

# START FILL ---------------------------------------------------------------
dataset = None  # Example: problem.dataset
sample_idx = 0
# design = ...         # np.array from dataset['train']['optimal_design'][sample_idx]
# config = ...         # dict over problem.conditions_keys
# END FILL -----------------------------------------------------------------

if dataset is None:
    raise RuntimeError("Set `dataset = problem.dataset` before running.")
if "design" not in locals() or "config" not in locals():
    raise RuntimeError("Define both `design` and `config` in the START FILL section.")

print(dataset)
print("sample_idx:", sample_idx)
print("design shape:", np.array(design).shape)
print("config:", config)

# CHECKPOINT
assert tuple(np.array(design).shape) == tuple(problem.design_space.shape), (
    f"design shape mismatch: expected {problem.design_space.shape}, got {np.array(design).shape}"
)
missing = [k for k in problem.conditions_keys if k not in config]
assert not missing, f"config missing condition keys: {missing}"
print("Checkpoint passed: dataset sample + config are valid.")

### Step 4 - Visualize one benchmark design

Render a sample and interpret what visual features correspond to feasible structure.


In [ ]:
# Render the sampled design (run after TODO 3)
fig, ax = problem.render(design)
ax.set_title("Participant: sampled Beams2D design")
plt.show()

### Step 5 - Test constraint semantics (PUBLIC FILL-IN)

Run a deliberate mismatch case.

Why this matters:
- robust benchmarking requires transparent failure modes,
- you should be able to explain *why* a design/config pair is invalid.

Success criteria:
- you can trigger and inspect at least one violation message.


In [ ]:
# PUBLIC FILL-IN CELL 00-C
# Goal: force a constraint mismatch and inspect violation diagnostics.

# START FILL ---------------------------------------------------------------
bad_config = dict(config)
# bad_config['volfrac'] = ...
# violations = ...
# END FILL -----------------------------------------------------------------

if "violations" not in locals():
    raise RuntimeError("Define `violations` in the START FILL section.")

print("Violation count:", len(violations))
if violations:
    for i, v in enumerate(violations[:5]):
        print(f"  [{i}]", v)
else:
    print("No violations found. Try a more aggressive volfrac mismatch.")

# CHECKPOINT
assert hasattr(violations, "__len__"), "violations should be a sized collection"
print("Checkpoint passed: constraint semantics inspected.")

## Troubleshooting

If a section fails, do not continue downstream. Fix locally first, then rerun the section and its immediate checks.
This notebook is intentionally staged so failures are localized.


## Next

Proceed to Notebook 01 to connect this benchmark interface to a concrete generative model pipeline.


## Reflection prompts

- Which benchmark fields must be reported in every paper for fair comparison?
- Which hidden defaults are most likely to create accidental unfairness?


## Takeaways

Before closing, record three points:
1. What conclusion is directly supported by your metrics?
2. What remains uncertain (and why)?
3. What extra experiment would you run next to reduce that uncertainty?
